# EXP_005 — Frozen REVE + Leadfield B_sim: PhysioNet EEGBCI (LOSO, 109 subjects)

Ports the EXP_003 result to **PhysioNet EEGBCI** — a much larger dataset than BCI IV 2a.
Answers: *does the +14.8% physics gain scale with more subjects and more channels?*

- 109 subjects, 64 channels, 160 Hz → 200 Hz
- Task: imagined left vs right fist (runs 6, 10, 14) — binary motor imagery
- Evaluation: leave-one-subject-out (LOSO), balanced accuracy
- Data: auto-downloaded via MNE — no local data needed.

**Speed tip:** set `FAST_MODE = True` to run only 20 subjects for a quick check.

In [ ]:
import subprocess, sys, os

IN_COLAB  = 'google.colab'  in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules or os.path.exists('/kaggle')

if IN_COLAB or IN_KAGGLE:
    repo = '/content/PhysREVE' if IN_COLAB else '/kaggle/working/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q',
            'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'mne>=1.6', 'transformers', 'huggingface_hub', 'einops', 'scipy'])
    print('Environment ready.')
else:
    print('Local environment.')

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings, json
warnings.filterwarnings('ignore')
import mne; mne.set_log_level('ERROR')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# Set to True for a quick 20-subject test run
FAST_MODE = False
print(f'Device: {DEVICE}  |  FAST_MODE: {FAST_MODE}')


## 2. Download PhysioNet EEGBCI + extract imagined movement epochs

In [ ]:
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
from mne import Epochs, events_from_annotations, pick_types

# Imagined left vs right fist: runs 6, 10, 14
RUNS         = [6, 10, 14]
TMIN, TMAX   = 0.0, 4.0          # epoch window (s)
SFREQ_ORIG   = 160.0
REVE_SFREQ   = 200.0
EVENT_ID     = {'T1': 0, 'T2': 1}   # T1=left, T2=right imagined fist

ALL_SUBJECTS = list(range(1, 110))   # 1–109
if FAST_MODE:
    ALL_SUBJECTS = ALL_SUBJECTS[:20]
    print(f'FAST_MODE: using {len(ALL_SUBJECTS)} subjects')

X_parts, y_parts, subj_parts = [], [], []
CH_NAMES = None

for s in ALL_SUBJECTS:
    try:
        fnames = eegbci.load_data(s, RUNS, verbose=False)
        raws   = [read_raw_edf(f, preload=True, verbose=False) for f in fnames]
        raw    = concatenate_raws(raws)
        eegbci.standardize(raw)   # rename channels to standard 10-20

        if CH_NAMES is None:
            CH_NAMES = [ch for ch in raw.ch_names if ch.startswith('E') is False]
            CH_NAMES = raw.ch_names   # use all 64

        events, _ = events_from_annotations(raw, event_id={'T1':1,'T2':2}, verbose=False)
        epochs     = Epochs(raw, events, tmin=TMIN, tmax=TMAX,
                            baseline=None, preload=True, verbose=False)
        epochs.drop_bad()

        Xs  = epochs.get_data(picks='eeg').astype(np.float32)   # (n_trials, 64, T)
        ys  = (epochs.events[:,2] - 1).astype(np.int64)         # 0=left, 1=right

        # Resample 160→200 Hz
        from scipy.signal import resample as scipy_resample
        n_t200 = int(Xs.shape[-1] * REVE_SFREQ / SFREQ_ORIG)
        Xs = scipy_resample(Xs, n_t200, axis=-1).astype(np.float32)

        X_parts.append(Xs); y_parts.append(ys)
        subj_parts.append(np.full(len(ys), s, dtype=np.int64))
    except Exception as e:
        print(f'  Subject {s} skipped: {e}')

X        = np.concatenate(X_parts)
y        = np.concatenate(y_parts)
subjects = np.concatenate(subj_parts)
UNIQUE_SUBJECTS = np.unique(subjects)
N_CH    = X.shape[1]
T_WIN   = X.shape[2]
CH_NAMES = [raw.ch_names[i] for i in range(N_CH)]  # grab final ch list

print(f'Loaded: {X.shape[0]} trials | {N_CH} channels | {T_WIN} samples @ {REVE_SFREQ:.0f} Hz')
print(f'Subjects: {len(UNIQUE_SUBJECTS)}  |  Classes: left={( y==0).sum()}  right={(y==1).sum()}')


## 3. Build leadfield for 64-channel 10-20 montage

In [ ]:
from physreve.physics import build_leadfield

print('Computing 64-channel leadfield (~60 s)...')
L_col_np, L_row_np, src_pos, lf_info = build_leadfield(
    ch_names=CH_NAMES, sfreq=REVE_SFREQ, verbose=True)

L_row_t  = torch.tensor(L_row_np, dtype=torch.float32).to(DEVICE)
B_sim    = (L_row_t @ L_row_t.T)
print(f'B_sim: {tuple(B_sim.shape)}  range [{B_sim.min():.2f}, {B_sim.max():.2f}]')


## 4. Load frozen REVE + electrode positions

In [ ]:
from huggingface_hub import notebook_login
from transformers import AutoModel

notebook_login()

print('Loading REVE position bank...')
pos_bank  = AutoModel.from_pretrained('brain-bzh/reve-positions', trust_remote_code=True)
POSITIONS = pos_bank(CH_NAMES)                   # (64, 3)
POSITIONS = POSITIONS.to(DEVICE)

print('Loading REVE-Base...')
reve_model = AutoModel.from_pretrained('brain-bzh/reve-base', trust_remote_code=True)
reve_model = reve_model.to(DEVICE).eval()
for p in reve_model.parameters():
    p.requires_grad = False
print(f'REVE frozen: {sum(p.numel() for p in reve_model.parameters()):,} params')

# Probe output
with torch.no_grad():
    _x = torch.randn(2, N_CH, T_WIN).to(DEVICE)
    _p = POSITIONS.unsqueeze(0).expand(2,-1,-1)
    _o = reve_model(_x, _p)
    _f = _o.last_hidden_state if hasattr(_o,'last_hidden_state') else (_o[0] if isinstance(_o,(tuple,list)) else _o)
print(f'REVE output: {_f.shape}')
REVE_FEAT_DIM = _f.mean(dim=list(range(1,_f.dim()-1))).shape[-1]
print(f'Feature dim: {REVE_FEAT_DIM}')
del _x, _p, _o, _f


## 5. Model

In [ ]:
class LeadfieldREVEExtractor(nn.Module):
    def __init__(self, reve_model, positions, B_sim,
                 sigma_elec=0.0, sigma_svd=0.0):
        super().__init__()
        self.reve       = reve_model
        self.sigma_elec = sigma_elec
        self.sigma_svd  = sigma_svd
        self.register_buffer('positions', positions)
        self.register_buffer('B_sim',     B_sim)
        self.alpha = nn.Parameter(torch.zeros(1))

    def _effective_B(self):
        if self.training and self.sigma_svd > 0:
            U, S, Vh = torch.linalg.svd(self.B_sim, full_matrices=False)
            S = S * (1.0 + self.sigma_svd * torch.randn_like(S))
            return U @ torch.diag(S) @ Vh
        return self.B_sim

    def forward(self, x):
        B_sz = x.size(0)
        pos  = self.positions
        if self.training and self.sigma_elec > 0:
            pos = pos + torch.randn_like(pos) * self.sigma_elec
        pos_b = pos.unsqueeze(0).expand(B_sz, -1, -1)
        with torch.no_grad():
            out  = self.reve(x, pos_b)
            feat = out.last_hidden_state if hasattr(out,'last_hidden_state') else (out[0] if isinstance(out,(tuple,list)) else out)
            if feat.dim() == 3: feat = feat.mean(dim=1)
        if feat.dim() == 4:
            B_eff = self._effective_B()
            return (feat + self.alpha * torch.einsum('ck,bkpd->bcpd', B_eff, feat)).mean(dim=(1,2))
        elif feat.dim() == 3:
            B_eff = self._effective_B()
            return (feat + self.alpha * torch.einsum('ck,bkd->bcd', B_eff, feat)).mean(dim=1)
        return feat


class MotorClassifier(nn.Module):
    def __init__(self, backbone, feat_dim, n_subjects, fusion_dim=128,
                 freeze_backbone=True, use_adversarial=False, adv_lambda=0.3):
        super().__init__()
        self.backbone        = backbone
        self.freeze_backbone = freeze_backbone
        self.use_adversarial = use_adversarial
        self.head = nn.Sequential(
            nn.Linear(feat_dim, fusion_dim), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(fusion_dim, fusion_dim), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(fusion_dim, 2),
        )
        if use_adversarial:
            from torch.autograd import Function
            class _GRL(Function):
                @staticmethod
                def forward(ctx, x, lam): ctx.lam=lam; return x.clone()
                @staticmethod
                def backward(ctx, g): return -ctx.lam*g, None
            self._grl = _GRL
            self.adv_lambda = adv_lambda
            self.subj_head  = nn.Sequential(
                nn.Linear(feat_dim, 64), nn.ELU(), nn.Linear(64, n_subjects))

    def forward(self, x):
        if self.freeze_backbone:
            with torch.no_grad(): f = self.backbone(x)
        else:
            f = self.backbone(x)
        logits = self.head(f)
        if self.use_adversarial and self.training:
            f_rev  = self._grl.apply(f, self.adv_lambda)
            return logits, self.subj_head(f_rev)
        return logits, None


backbone_A = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.0,   0.0  ).to(DEVICE)
backbone_B = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.0,   0.0  ).to(DEVICE)
backbone_C = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.005, 0.10 ).to(DEVICE)
CONDITION_BACKBONES = {
    'A_baseline':      (backbone_A, True),
    'B_leadfield':     (backbone_B, False),
    'C_leadfield_aug': (backbone_C, False),
}
print('Backbones built.')


## 6. Data prep

In [ ]:
# Z-score per trial
X_norm = X.copy()
X_norm -= X_norm.mean(axis=-1, keepdims=True)
X_norm /= (X_norm.std(axis=-1, keepdims=True) + 1e-8)
print(f'Normalised EEG: {X_norm.shape}')


class EEGDataset(Dataset):
    def __init__(self, X, y, subj_ids):
        self.X    = torch.FloatTensor(X)
        self.y    = torch.LongTensor(y)
        self.subj = torch.LongTensor(subj_ids)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i], self.subj[i]


def train_epoch(model, loader, opt, n_train_subj, device):
    model.train()
    if model.freeze_backbone: model.backbone.eval()
    preds, labels = [], []
    for Xb, yb, sb in loader:
        Xb, yb, sb = Xb.to(device), yb.to(device), sb.to(device)
        opt.zero_grad()
        task_logits, subj_logits = model(Xb)
        loss = F.cross_entropy(task_logits, yb)
        if subj_logits is not None:
            loss = loss + 0.3 * F.cross_entropy(subj_logits, sb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        preds.extend(task_logits.argmax(1).cpu().numpy())
        labels.extend(yb.cpu().numpy())
    return balanced_accuracy_score(labels, preds)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    for Xb, yb, _ in loader:
        Xb = Xb.to(device)
        logits, _ = model(Xb)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(yb.numpy())
    return balanced_accuracy_score(np.array(labels), np.array(preds))


## 7. Leave-One-Subject-Out cross-validation

In [ ]:
BATCH_SIZE   = 32
MAX_EPOCHS   = 80
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 15

all_results = {cond: {} for cond in CONDITION_BACKBONES}

print('='*65)
print('REVE + B_sim  |  PhysioNet EEGBCI  |  LOSO')
print(f'Subjects: {len(UNIQUE_SUBJECTS)}  |  Conditions: A / B / C')
print('='*65)

for test_subj in UNIQUE_SUBJECTS:
    test_mask  = subjects == test_subj
    train_mask = ~test_mask

    le = LabelEncoder()
    train_subj_enc = le.fit_transform(subjects[train_mask])

    train_ds = EEGDataset(X_norm[train_mask], y[train_mask], train_subj_enc)
    test_ds  = EEGDataset(X_norm[test_mask],  y[test_mask],
                          np.zeros(test_mask.sum(), dtype=np.int64))
    train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True)
    test_dl  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False)

    subj_line = f'S{test_subj:03d}'
    for cond_name, (bb, freeze_bb) in CONDITION_BACKBONES.items():
        model = MotorClassifier(
            bb, REVE_FEAT_DIM,
            n_subjects=len(le.classes_),
            freeze_backbone=freeze_bb,
            use_adversarial=False,     # keep same as EXP_003 for clean comparison
        ).to(DEVICE)
        opt   = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, MAX_EPOCHS)

        best_acc, patience_ctr = 0, 0
        for epoch in range(MAX_EPOCHS):
            train_epoch(model, train_dl, opt, len(le.classes_), DEVICE)
            sched.step()
            if (epoch+1) % 5 == 0 or epoch == 0:
                acc = evaluate(model, test_dl, DEVICE)
                if acc > best_acc:
                    best_acc = acc; patience_ctr = 0
                else:
                    patience_ctr += 5
                if patience_ctr >= PATIENCE: break

        all_results[cond_name][int(test_subj)] = float(best_acc)
        del model, opt, sched
        torch.cuda.empty_cache()

    # Print one line per subject
    a = all_results['A_baseline'][int(test_subj)]
    b = all_results['B_leadfield'][int(test_subj)]
    c = all_results['C_leadfield_aug'][int(test_subj)]
    print(f'{subj_line}  A={a:.3f}  B={b:.3f}  C={c:.3f}  ΔB={b-a:+.3f}')


## 8. Results

In [ ]:
print('\n' + '='*65)
print('PhysioNet EEGBCI — LOSO Summary')
print('='*65)
for cond in ['A_baseline','B_leadfield','C_leadfield_aug']:
    vals = list(all_results[cond].values())
    b_vals = list(all_results['A_baseline'].values())
    wins   = sum(v > b for v,b in zip(vals, b_vals))
    print(f'  {cond:22s}  mean={np.mean(vals):.3f}±{np.std(vals):.3f}  '
          f'wins vs A: {wins}/{len(vals)}')

print()
delta_b = np.mean(list(all_results['B_leadfield'].values())) - \
          np.mean(list(all_results['A_baseline'].values()))
print(f'B vs A improvement: {delta_b:+.3f}  ({delta_b/np.mean(list(all_results["A_baseline"].values()))*100:+.1f}%)')
print()
print('Comparison with EXP_003 (BCI IV-2a, 9 subjects):')
print('  A baseline:  0.338 ± 0.045')
print('  B +B_sim:    0.388 ± 0.079  (+14.8%)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PhysioNet EEGBCI — Frozen REVE + B_sim  |  LOSO', fontsize=13)

conds  = ['A_baseline','B_leadfield','C_leadfield_aug']
labels = ['A — Baseline','B — +B_sim','C — B+jitter']
colors = ['#85B7EB','#5DCAA5','#AFA9EC']

subj_list = sorted(all_results['A_baseline'].keys())
a_vals = [all_results['A_baseline'][s]      for s in subj_list]
b_vals = [all_results['B_leadfield'][s]     for s in subj_list]
c_vals = [all_results['C_leadfield_aug'][s] for s in subj_list]
means  = [np.mean(a_vals), np.mean(b_vals), np.mean(c_vals)]
stds   = [np.std(a_vals),  np.std(b_vals),  np.std(c_vals)]

ax = axes[0]
bars = ax.bar(labels, means, color=colors, edgecolor='white', width=0.5)
ax.errorbar(labels, means, yerr=stds, fmt='none', color='black', capsize=5)
ax.axhline(0.5, color='red', lw=1, ls='--', label='Chance')
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{m:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Mean Balanced Accuracy'); ax.set_title('Mean ± Std across subjects')
ax.set_ylim(0, min(1.0, max(means)*1.3)); ax.grid(alpha=0.3, axis='y'); ax.legend()

ax = axes[1]
ax.scatter(a_vals, np.array(b_vals)-np.array(a_vals),
           s=30, c='#5DCAA5', alpha=0.6, label='+B_sim', edgecolors='none')
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Baseline accuracy (frozen REVE)')
ax.set_ylabel('ΔAUC vs baseline')
ax.set_title(f'Per-subject gain  (n={len(subj_list)})'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exp005_physionet_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
with open('exp005_results.json', 'w') as f:
    json.dump({'experiment': 'EXP_005', 'backbone': 'REVE-Base',
               'dataset': 'PhysioNet-EEGBCI', 'evaluation': 'LOSO',
               'n_subjects': len(subj_list),
               'results': {k: {str(s): float(v) for s,v in d.items()}
                           for k,d in all_results.items()}}, f, indent=2)
print('Saved exp005_results.json')
